# Unwrap the Insights — SQL analysis and reporting model
## Run in Colab
Save this notebook in `notebooks/Ferrero_SQL_Analysis.ipynb` in your repository and open it with Colab. Run top to bottom after the preparation notebook. Uses Python's built-in SQLite engine; no MySQL installation or credentials required.

This notebook reads `data/processed/ferrero_sales_clean.csv`. It creates a star model, runs SQL and exports results. It **does not automatically commit outputs to GitHub**. Commit reviewed files separately. Original cleaning files are unchanged.

Independent portfolio analysis. Currency and real-world provenance are unverified. Default analysis is Australia; do not label currency AUD or treat cross-country monetary sums as comparable.

In [1]:
from pathlib import Path
import sqlite3
import subprocess
import pandas as pd
import numpy as np

# Reuse the repository cloned by the preparation notebook in this Colab session.
REPO_DIR = Path('/content/ferrero-rocher-sales-analytics')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/milaho1006/ferrero-rocher-sales-analytics.git', str(REPO_DIR)], check=True)
source = REPO_DIR / 'data/processed/ferrero_sales_clean.csv'
if not source.exists():
    raise FileNotFoundError('Run the preparation notebook first, or commit its cleaned CSV to data/processed/.')
OUTPUT_DIR = REPO_DIR / 'analysis/sql_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean = pd.read_csv(source)
assert not clean.isna().any().any()
assert clean['Source Row Number'].is_unique
COUNTRY = 'Australia'  # Change to another supplied country, then rerun analysis cells.
assert COUNTRY in clean['Country'].unique()
# Rebuild in memory on each run; the completed database is exported later.
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON')
print(f'Loaded {len(clean):,} records; selected country: {COUNTRY}')

Loaded 20,000 records; selected country: Australia


## 1. Reporting model
**Grain:** one supplied sales record. Source Row Number preserves CSV lineage, not an order ID. The fact table holds quantities and financial measures; dimensions hold descriptive attributes. Each dimension has one row per distinct member (date dimension also has calendar attributes).

Retailer names represent chains, not physical stores. Country is a separate dimension because a retailer can appear in multiple countries. Promotion name is a dimension; discount remains on each sales row. We do not assume fixed discounts as a permanent business rule.

All relationships run from one dimension member to many sales records. Unique keys and foreign keys prevent ambiguous links.

In [2]:
# Dimensions use integer surrogate keys. Keys are deterministic for the same input.
specs = [
 ('dim_date', 'date_id', ['Date','Year','Month Number','Month Start','Quarter'], ['sale_date','year','month_number','month_start','quarter']),
 ('dim_product', 'product_id', ['SKU'], ['sku']),
 ('dim_country', 'country_id', ['Country'], ['country']),
 ('dim_retailer', 'retailer_id', ['Store'], ['retailer']),
 ('dim_channel', 'channel_id', ['Channel'], ['channel']),
 ('dim_promotion', 'promotion_id', ['Promotion'], ['promotion'])
]
fact = clean.copy()
for table, key, cols, names in specs:
    dimension = clean[cols].drop_duplicates().sort_values(cols).reset_index(drop=True)
    dimension.insert(0, key, range(1, len(dimension)+1))
    types = ['INTEGER' if name in ['year','month_number'] else 'TEXT' for name in names]
    definitions = ', '.join(f'{name} {dtype} NOT NULL' for name,dtype in zip(names,types))
    conn.execute(f'CREATE TABLE {table} ({key} INTEGER PRIMARY KEY, {definitions}, UNIQUE ({names[0]}))')
    dimension.rename(columns=dict(zip(cols,names))).to_sql(table,conn,if_exists='append',index=False)
    fact = fact.merge(dimension[cols+[key]], on=cols, how='left', validate='many_to_one')
conn.execute("""CREATE TABLE fact_sales (
 source_row_number INTEGER PRIMARY KEY,
 date_id INTEGER NOT NULL REFERENCES dim_date(date_id),
 product_id INTEGER NOT NULL REFERENCES dim_product(product_id),
 country_id INTEGER NOT NULL REFERENCES dim_country(country_id),
 retailer_id INTEGER NOT NULL REFERENCES dim_retailer(retailer_id),
 channel_id INTEGER NOT NULL REFERENCES dim_channel(channel_id),
 promotion_id INTEGER NOT NULL REFERENCES dim_promotion(promotion_id),
 units_sold INTEGER NOT NULL CHECK (units_sold > 0),
 unit_price REAL NOT NULL CHECK (unit_price > 0),
 discount REAL NOT NULL CHECK (discount BETWEEN 0 AND 1),
 revenue REAL NOT NULL,
 margin_percent REAL NOT NULL CHECK (margin_percent BETWEEN 0 AND 100),
 margin_amount REAL NOT NULL
)""")
rename = {'Source Row Number':'source_row_number','Units Sold':'units_sold','Unit Price':'unit_price',
          'Discount':'discount','Revenue':'revenue','Margin %':'margin_percent','Margin':'margin_amount'}
keys = [key for _,key,_,_ in specs]
fact[['Source Row Number']+keys+['Units Sold','Unit Price','Discount','Revenue','Margin %','Margin']].rename(columns=rename).to_sql('fact_sales',conn,if_exists='append',index=False)
conn.execute("""CREATE VIEW sales_reporting AS
 SELECT f.*, d.sale_date, d.year, d.month_number, d.month_start, d.quarter,
 p.sku, c.country, r.retailer, h.channel, m.promotion
 FROM fact_sales f
 JOIN dim_date d USING(date_id) JOIN dim_product p USING(product_id)
 JOIN dim_country c USING(country_id) JOIN dim_retailer r USING(retailer_id)
 JOIN dim_channel h USING(channel_id) JOIN dim_promotion m USING(promotion_id)""")
conn.commit()
print('Created six dimensions, one sales fact table and a reporting view.')

Created six dimensions, one sales fact table and a reporting view.


## 2. Reconcile SQL against the clean data
Check the full reporting join as well as per-country quantities and financial totals. This catches accidental row multiplication or dropped records before interpretation.

In [3]:
# Confirm joins did not duplicate/drop sales and every country reconciles.
assert conn.execute('PRAGMA foreign_key_check').fetchall() == []
assert conn.execute('SELECT COUNT(*) FROM sales_reporting').fetchone()[0] == len(clean)
sql_totals = pd.read_sql_query('SELECT country, COUNT(*) AS records, SUM(units_sold) AS units, SUM(revenue) AS revenue, SUM(margin_amount) AS margin FROM sales_reporting GROUP BY country ORDER BY country',conn)
py_totals = clean.groupby('Country',sort=True).agg(records=('Revenue','size'),units=('Units Sold','sum'),revenue=('Revenue','sum'),margin=('Margin','sum')).reset_index()
assert sql_totals['country'].tolist() == py_totals['Country'].tolist()
assert np.allclose(sql_totals[['records','units','revenue','margin']],py_totals[['records','units','revenue','margin']],rtol=1e-10,atol=0.01)
print('Foreign keys, joined row count and all country totals reconcile with pandas.')
sql_totals[['country','records','units']]

Foreign keys, joined row count and all country totals reconcile with pandas.
  country  records  units
Australia     2777 288641
   Canada     2958 310080
    India     2740 288419
Singapore     2846 299694
      UAE     2818 296329
       UK     3010 312378
      USA     2851 297333


## 3. KPI definitions
| Measure | Calculation | Interpretation |
|---|---|---|
| Sales records | COUNT(*) | Supplied rows, not orders |
| Units sold | SUM(units_sold) | Product packs/units as supplied, not individual chocolates |
| Recorded revenue | SUM(revenue) | Unspecified dataset currency units |
| Supplied margin | SUM(margin_amount) | Supplied margin, not verified gross/net profit |
| Weighted margin rate | SUM(margin_amount)/SUM(revenue) | Format as percentage |
| Units per record | SUM(units_sold)/COUNT(*) | Descriptive volume per supplied row |
| Weighted discount | SUM(units × price × discount)/SUM(units × price) | Share of pre-discount recorded value |

SQL uses NULLIF to avoid division by zero. Rates are decimal fractions. Month-over-month change uses LAG; the first month is NULL. Review date coverage before interpreting changes.

In [4]:
queries = {'01_kpis': 'SELECT COUNT(*) AS sales_records, SUM(units_sold) AS units_sold,\n SUM(revenue) AS recorded_revenue, SUM(margin_amount) AS supplied_margin,\n SUM(margin_amount)/NULLIF(SUM(revenue),0) AS weighted_margin_rate,\n 1.0*SUM(units_sold)/NULLIF(COUNT(*),0) AS units_per_record,\n SUM(units_sold*unit_price*discount)/NULLIF(SUM(units_sold*unit_price),0) AS weighted_discount\nFROM sales_reporting WHERE country = :country', '02_product_mix': 'SELECT sku, COUNT(*) AS sales_records, SUM(units_sold) AS units_sold,\n SUM(revenue) AS recorded_revenue,\n SUM(revenue)/NULLIF(SUM(SUM(revenue)) OVER (),0) AS revenue_share,\n SUM(margin_amount)/NULLIF(SUM(revenue),0) AS weighted_margin_rate\nFROM sales_reporting WHERE country = :country\nGROUP BY sku ORDER BY recorded_revenue DESC', '03_monthly_trend': 'WITH monthly AS (\n SELECT month_start, SUM(revenue) AS recorded_revenue, SUM(units_sold) AS units_sold\n FROM sales_reporting WHERE country = :country GROUP BY month_start\n), previous AS (\n SELECT *, LAG(recorded_revenue) OVER (ORDER BY month_start) AS previous_month_revenue\n FROM monthly\n)\nSELECT *, (recorded_revenue-previous_month_revenue)/NULLIF(previous_month_revenue,0) AS revenue_mom_change\nFROM previous ORDER BY month_start', '04_promotion_comparison': 'SELECT promotion, discount, COUNT(*) AS sales_records,\n SUM(units_sold) AS units_sold, AVG(units_sold) AS units_per_record,\n SUM(revenue) AS recorded_revenue, AVG(revenue) AS revenue_per_record,\n SUM(margin_amount)/NULLIF(SUM(revenue),0) AS weighted_margin_rate\nFROM sales_reporting WHERE country = :country\nGROUP BY promotion, discount ORDER BY discount', '05_promotion_within_segments': 'SELECT sku, channel, promotion, discount, COUNT(*) AS sales_records,\n AVG(units_sold) AS units_per_record, AVG(revenue) AS revenue_per_record\nFROM sales_reporting WHERE country = :country\nGROUP BY sku, channel, promotion, discount\nORDER BY sku, channel, discount', '06_channel_performance': 'SELECT channel, COUNT(*) AS sales_records, SUM(units_sold) AS units_sold,\n SUM(revenue) AS recorded_revenue,\n SUM(margin_amount)/NULLIF(SUM(revenue),0) AS weighted_margin_rate\nFROM sales_reporting WHERE country = :country\nGROUP BY channel ORDER BY recorded_revenue DESC', '07_retailer_performance': 'SELECT retailer, COUNT(*) AS sales_records, SUM(units_sold) AS units_sold,\n SUM(revenue) AS recorded_revenue,\n SUM(margin_amount)/NULLIF(SUM(revenue),0) AS weighted_margin_rate\nFROM sales_reporting WHERE country = :country\nGROUP BY retailer ORDER BY recorded_revenue DESC'}
results = {}
params = {"country": COUNTRY}

### 01 kpis
Overall KPIs establish a reconciliation baseline for Tableau.

In [5]:
results['01_kpis'] = pd.read_sql_query(queries['01_kpis'], conn, params=params)
results['01_kpis']

 sales_records  units_sold  recorded_revenue  supplied_margin  weighted_margin_rate  units_per_record  weighted_discount
          2777      288641       110372016.5       35876636.5              0.325052        103.939863           0.205593


### 02 product mix
Compare revenue share with volume. A high-priced pack can lead revenue without leading unit volume.

In [6]:
results['02_product_mix'] = pd.read_sql_query(queries['02_product_mix'], conn, params=params)
results['02_product_mix']

                    sku  sales_records  units_sold  recorded_revenue  revenue_share  weighted_margin_rate
    Ferrero Rocher 32pc            538       54163        31983375.0       0.289778              0.324979
Ferrero Collection 15pc            552       58763        30354707.5       0.275022              0.324208
    Ferrero Rocher 24pc            572       58888        25899472.5       0.234656              0.326273
    Ferrero Rocher 16pc            564       59611        16574057.5       0.150165              0.325144
      Ferrero Rocher T3            551       57216         5560404.0       0.050379              0.324112


### 03 monthly trend
This is a within-year trend, not proof of recurring seasonality. Month totals also reflect different numbers of days and records.

In [7]:
results['03_monthly_trend'] = pd.read_sql_query(queries['03_monthly_trend'], conn, params=params)
results['03_monthly_trend']

month_start  recorded_revenue  units_sold  previous_month_revenue  revenue_mom_change
 2025-01-01         9833901.5       25746                     NaN                 NaN
 2025-02-01         7417677.5       19796               9833901.5           -0.245703
 2025-03-01         9266895.0       23985               7417677.5            0.249299
 2025-04-01         8469411.0       22178               9266895.0           -0.086057
 2025-05-01         9351809.5       24374               8469411.0            0.104187
 2025-06-01         9389119.0       23984               9351809.5            0.003990
 2025-07-01         9821862.5       24571               9389119.0            0.046090
 2025-08-01        10143073.5       28054               9821862.5            0.032704
 2025-09-01         8213042.5       22706              10143073.5           -0.190281
 2025-10-01         9787946.5       24464               8213042.5            0.191756
 2025-11-01         9101844.0       23445             

### 04 promotion comparison
Compare counts and units per record alongside totals. Promotion names and discounts are confounded; no causal uplift or ROI claim is supported.

In [8]:
results['04_promotion_comparison'] = pd.read_sql_query(queries['04_promotion_comparison'], conn, params=params)
results['04_promotion_comparison']

      promotion  discount  sales_records  units_sold  units_per_record  recorded_revenue  revenue_per_record  weighted_margin_rate
   No promotion      0.00            433       45123        104.210162        20486000.0        47311.778291              0.326732
Valentine Promo      0.10            458       45930        100.283843        20057652.0        43794.000000              0.331064
   Diwali Promo      0.15            478       49389        103.324268        20451697.0        42785.976987              0.322986
Christmas Promo      0.20            470       51010        108.531915        19649080.0        41806.553191              0.325541
   NewYear Sale      0.25            478       49128        102.778243        17869882.5        37384.691423              0.319674
       Buy1Get1      0.50            460       48061        104.480435        11857705.0        25777.619565              0.322838


### 05 promotion within segments
Compare within the same SKU and channel to reduce observed mix differences. Small groups are visible through sales_records; this still is not a controlled experiment.

In [9]:
results['05_promotion_within_segments'] = pd.read_sql_query(queries['05_promotion_within_segments'], conn, params=params)
results['05_promotion_within_segments']

                    sku   channel       promotion  discount  sales_records  units_per_record  revenue_per_record
Ferrero Collection 15pc    Online    No promotion      0.00             21         99.666667        64783.333333
Ferrero Collection 15pc    Online Valentine Promo      0.10             29         93.862069        54909.310345
Ferrero Collection 15pc    Online    Diwali Promo      0.15             32        111.531250        61621.015625
Ferrero Collection 15pc    Online Christmas Promo      0.20             25        100.720000        52374.400000
Ferrero Collection 15pc    Online    NewYear Sale      0.25             31        109.354839        53310.483871
Ferrero Collection 15pc    Online        Buy1Get1      0.50             34        118.147059        38397.794118
Ferrero Collection 15pc    Retail    No promotion      0.00             31         92.967742        60429.032258
Ferrero Collection 15pc    Retail Valentine Promo      0.10             25        116.640000    

### 06 channel performance
Channel labels may overlap conceptually (Online/Offline versus Retail/Wholesale). Preserve them and request definitions before a business decision.

In [10]:
results['06_channel_performance'] = pd.read_sql_query(queries['06_channel_performance'], conn, params=params)
results['06_channel_performance']

  channel  sales_records  units_sold  recorded_revenue  weighted_margin_rate
Wholesale            964       98846        38087513.0              0.327685
   Retail            920       98211        37855416.5              0.324270
   Online            893       91584        34429087.0              0.322999


### 07 retailer performance
Retailer totals are within the selected country. Do not interpret chain labels as individual store locations.

In [11]:
results['07_retailer_performance'] = pd.read_sql_query(queries['07_retailer_performance'], conn, params=params)
results['07_retailer_performance']

  retailer  sales_records  units_sold  recorded_revenue  weighted_margin_rate
Woolworths            399       42466        16529465.5              0.328547
     Kmart            417       43429        16314573.0              0.322369
     Tesco            402       41187        15837756.0              0.324723
  7-Eleven            389       39932        15548051.5              0.328040
   Walmart            380       39993        15544812.0              0.325501
      Aldy            393       40529        15399451.5              0.323801
      BigW            270       28403        10426582.5              0.324066
     Coles            127       12702         4771324.5              0.318202


## 4. Export reproducible outputs
CSV results are saved under a country-specific folder. The complete database contains all countries, with no credentials. Tableau can continue using the original cleaned CSV; these SQL exports are reconciliation and analysis evidence, not replacements for the row-level source.

In [12]:
country_dir = OUTPUT_DIR / COUNTRY.lower().replace(' ', '_')
country_dir.mkdir(exist_ok=True)
for name, frame in results.items():
    frame.to_csv(country_dir / f'{name}.csv', index=False)
sql_script = '-- SQLite; bind :country to one supplied country.\n\n' + '\n\n'.join('-- '+name+'\n'+query+';' for name,query in queries.items())
sql_dir = REPO_DIR / 'sql'
sql_dir.mkdir(exist_ok=True)
(sql_dir / 'sales_analysis.sql').write_text(sql_script)
conn.commit()
with sqlite3.connect(OUTPUT_DIR / 'ferrero_sales.sqlite') as destination:
    conn.backup(destination)
print('Saved query results:', country_dir)
print('Saved SQL:', sql_dir / 'sales_analysis.sql')
print('Saved model:', OUTPUT_DIR / 'ferrero_sales.sqlite')

Saved query results: /workspace/scratch/5e739c2e1162/ferrero_step2/analysis/sql_outputs/australia
Saved SQL: /workspace/scratch/5e739c2e1162/ferrero_step2/sql/sales_analysis.sql
Saved model: /workspace/scratch/5e739c2e1162/ferrero_step2/analysis/sql_outputs/ferrero_sales.sqlite


## Next: interpret, then build Tableau
Review the product and promotion outputs before writing findings. For every finding record: numeric evidence, business implication, suggested action, and limitation. Confirm the selected country in every chart. Tableau must match SQL KPIs under the same filters.

Save the notebook to GitHub using Colab's Save a copy in GitHub. Generated CSV/SQL/database files require separate reviewed commits; saving the notebook alone does not commit them. The SQLite binary is optional in GitHub since the notebook rebuilds it.